# Infra-Bench CLS — OlmoEarth v1.1-Base Linear Probe

Linear-probe evaluation of OlmoEarth v1.1-Base on the Infra-Bench CLS
13-class benchmark. Frozen backbone, trainable linear head, 3 seeds,
spatial split.

## Model

- **OlmoEarth v1.1-Base** (Allen Institute for AI, 2024). ViT-Base
  multimodal Earth-observation FM.
- Loader:
  `olmoearth_pretrain_minimal.load_model_from_id(ModelID.OLMOEARTH_V1_1_BASE)`.
  Falls back to v1-Base if the v1.1 `ModelID` enum is not exposed by the
  pinned release.
- Feature: mean-pooled patch features, dim 768.

## Input pipeline

### Band selection
Our `.npy` storage: `[B04, B03, B02, B08, B8A, B11, B12, VV, VH]`.

OlmoEarth's S2 interface expects **12 bands**. We supply 6 (B02, B03,
B04, B08, B8A, B11, B12) and **zero-pad the other 6** (B01, B05, B06,
B07, B09, B10). Relies on v1.1's band-dropout robustness during
pretraining.

Input reshaped to `(B, H, W, T=1, 12)` inside the backbone's
`_prepare_input`.

### Normalization
OlmoEarth's own `Normalizer(Modality.SENTINEL2_L2A, ...)` applied inside
the backbone's `_prepare_input` — not the shared `percentile_normalize`.

### Image size
`IMAGE_SIZE = 224`.

## Split (shared across all Infra-Bench CLS FMs)

Spatial block-based split loaded from
`data/spatial_split/asset_id_to_split_v1.parquet`. Blocks assign a whole
~0.5° spatial region to the same partition. Invariant across seeds and
across every FM.

## Training protocol

- **25 epochs**, batch 16, AdamW, LR **1e-3**, weight decay **1e-4**
- Class-weighted CE, weights capped at 10×
- **Frozen backbone** (`freeze=True`). Head is `Linear(768 → 13)` with
  `Dropout(0.1)`.
- **3 seeds** (314, 271, 161) varying head init + DataLoader shuffle only.
- **Best-val checkpoint restored before the held-out test pass**.

## Per-sector F1 definition

Macro-average of per-class F1s for classes in that sector, computed on
the FULL test set. Return schema: `{n, macro_f1, acc,
per_class_f1_in_sector}` per sector.

## Aggregate output

Combines the 3 seeds into `mean ± std` and `per_seed` arrays. Write is
gated by `set(SEEDS) == set(FULL_PROTOCOL_SEEDS)`.

## Outputs

- Per-seed: `results/fm_eval_olmoearth_lp_v1/olmoearth_lp_v1_seed{314,271,161}_results.json`
- Aggregate: `results/fm_eval_olmoearth_lp_v1/olmoearth_lp_v1_aggregate.json`
- Confusion matrix: `results/fm_eval_olmoearth_lp_v1/confusion_matrix_olmoearth_lp_v1_aggregate.png`

## Runtime

`SMOKE_ONLY=True` by default.


In [ ]:
# Colab's base image already has a coherent torch/torchvision/torchaudio triple;
# force-reinstalling torch breaks torchaudio ABI. olmoearth_pretrain_minimal
# requires torch>=2.7 which Colab's default already satisfies. don't touch
# torch/torchvision/numpy -- let Colab's base image provide the CUDA+ABI-
# consistent tuple.
!pip install -q olmoearth_pretrain_minimal
!pip install -q pyarrow scikit-learn scipy huggingface-hub==0.36.2

# verify torch/torchvision loaded cleanly against the current CUDA
!python -c "import torch, torchvision; print(f'torch={torch.__version__}, torchvision={torchvision.__version__}, cuda_ok={torch.cuda.is_available()}')"

In [ ]:
# pre-flight: check what's actually on disk for the critical libraries.
# pip queries the filesystem, not Python's in-memory imports, so this
# tells us the truth regardless of what's loaded.
import subprocess

critical = ['torch', 'torchvision', 'pillow', 'numpy', 'scipy',
            'scikit-learn', 'olmoearth_pretrain_minimal',
            'huggingface-hub', 'pyarrow']

for pkg in critical:
    result = subprocess.run(
        ['pip', 'show', pkg],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        # parse the "Version: X.Y.Z" line
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                print(f'  {pkg:<20s} {line.split(":", 1)[1].strip()}')
                break
    else:
        print(f'  {pkg:<20s} NOT INSTALLED')

In [ ]:
# conditional numpy restore. some Colab images resolve pip installs
# in a way that pins numpy down to 1.x, which then breaks sklearn 1.5+.
# only force the upgrade if we detect the downgrade. when we do upgrade,
# raise to force a clean kernel restart -- pip cannot swap numpy in an
# already-loaded kernel.
import numpy as np
if np.__version__.startswith('1.'):
    print(f'numpy is on {np.__version__} (1.x). Restoring to 2.x and forcing a restart...')
    !pip install --upgrade --force-reinstall "numpy>=2.2"
    !pip install --upgrade --force-reinstall --no-deps scikit-learn scipy
    raise RuntimeError(
        'numpy was pinned to 1.x by the install above; restored to 2.x. '
        'Restart the Colab kernel (Runtime -> Restart session) and re-run '
        'from the top.'
    )
else:
    print(f'numpy {np.__version__} OK (no restore needed)')

# also re-verify sklearn/scipy import cleanly after the install step.
import sklearn, scipy
print(f'sklearn  {sklearn.__version__}')
print(f'scipy    {scipy.__version__}')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Runtime -> Change runtime type -> GPU before training.')


In [ ]:
import torch
print(f"torch: {torch.__version__}")
print(f"torchvision available:", end=" ")
try:
    import torchvision
    print(f"{torchvision.__version__}")
except Exception as e:
    print(f"FAILED — {e}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
# verify nms operator (the one that was failing)
try:
    from torchvision.ops import nms
    print("torchvision.ops.nms: OK")
except Exception as e:
    print(f"torchvision.ops.nms: FAILED — {e}")

In [ ]:
# ============================================================================
# HF authentication -- resilient chain: Colab Secrets -> .env on Drive -> env
# ----------------------------------------------------------------------------
# the token is only a fallback for HF metadata calls. primary weight load
# path is the Drive HF cache with HF_HUB_OFFLINE=1, so "no token found"
# is not a failure in this notebook.
# ============================================================================
import os

hf_token = None

# 1. Colab Secrets (works when notebook is running interactively in the UI)
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        print('  HF_TOKEN found via Colab Secrets')
except Exception:
    pass  # userdata unavailable (background exec, timeout, non-Colab, etc.)

# 2. .env on Drive (portable across sessions; survives runtime restarts)
if not hf_token:
    try:
        from dotenv import load_dotenv
    except ImportError:
        import subprocess
        subprocess.run(['pip', 'install', '-q', 'python-dotenv'], check=False)
        from dotenv import load_dotenv
    for env_path in ['/content/drive/MyDrive/infra_fm/.env',
                     '/content/drive/MyDrive/.env']:
        if os.path.exists(env_path):
            load_dotenv(env_path)
            hf_token = os.environ.get('HF_TOKEN')
            if hf_token:
                print(f'  HF_TOKEN found via {env_path}')
                break

# 3. any HF_TOKEN already in os.environ (e.g. set by an earlier cell)
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        print('  HF_TOKEN found via os.environ')

# 4. login if we got a token; otherwise proceed anonymously (Drive cache carries us)
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in to HuggingFace.')
else:
    print('No HF_TOKEN via userdata / .env / environ -- proceeding anonymously.')
    print('(Fine when HF_HUB_OFFLINE=1 and models are pre-loaded in Drive cache.)')


In [ ]:
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infrabench_repo'

if not Path(f'{EXTRACT_TO}/curation').exists():
    if Path(CODE_ZIP).exists():
        print(f'Extracting {CODE_ZIP} -> {EXTRACT_TO} ...')
        os.makedirs(EXTRACT_TO, exist_ok=True)
        with zipfile.ZipFile(CODE_ZIP, 'r') as z:
            for member in z.namelist():
                clean = member.replace('\\', '/')
                target = os.path.join(EXTRACT_TO, clean)
                if clean.endswith('/'):
                    os.makedirs(target, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    with z.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
        print('done.')

for candidate in [EXTRACT_TO, f'{EXTRACT_TO}/infra_fm_code_only']:
    if Path(candidate).exists() and candidate not in sys.path:
        sys.path.insert(0, candidate)

# only load_split_artifact is needed at training time. if the curation
# zip predates Phase 1, fall back to an inline definition.
try:
    from curation.utils.spatial_blocking import load_split_artifact
    print('Imported load_split_artifact from curation.utils.spatial_blocking')
except ImportError as e:
    print(f'Could not import (zip is pre-Phase-1): {e}. Using inline fallback.')
    import pandas as pd
    def load_split_artifact(input_path):
        df = pd.read_parquet(input_path, columns=['asset_id', 'split'])
        return dict(zip(df['asset_id'].astype(str), df['split'].astype(str)))


In [ ]:
import os
from pathlib import Path

DATASETS_DRIVE      = f'{DRIVE_ROOT}/datasets'
DATASETS_LOCAL      = '/content/datasets'
OUTPUT_DIR          = f'{DRIVE_ROOT}/results/fm_eval_olmoearth_lp_v1'
# the split artifact ships inside the code zip, so it resolves from the
# extracted repo. Drive stays a fallback for setups that still stage it there.
_drive_split = f'{DRIVE_ROOT}/data/spatial_split/asset_id_to_split_v1.parquet'
try:
    from curation.paths import SPLIT_ARTIFACT as _repo_split
    SPLIT_ARTIFACT_PATH = str(_repo_split) if _repo_split.exists() else _drive_split
except Exception:
    SPLIT_ARTIFACT_PATH = _drive_split
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATASETS_LOCAL, exist_ok=True)

REGIONS = [
    'africa', 'asia', 'australia-oceania', 'central-america',
    'europe', 'north-america', 'south-america',
]
SECTORS = ['energy', 'water', 'transport', 'telecom']

CLASS_NAMES = [
    'energy.transmission.substation',
    'energy.distribution.substation',
    'energy.distribution.other',
    'energy.generation.power_plant',
    'energy.generation.solar_farm',
    'energy.generation.wind_farm',
    'water.wastewater.plant',
    'water.water_works',
    'water.storage_tank',
    'transport.airport',
    'transport.train_station',
    'transport.port_terminal',
    'telecom.data_center',
]
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}
ASSET_TYPE_MAP = {
    'energy.transmission.substation':         'energy.transmission.substation',
    'energy.distribution.substation':         'energy.distribution.substation',
    'energy.distribution.substation_untyped': 'energy.distribution.other',
    'energy.distribution.substation_minor':   'energy.distribution.other',
    'energy.generation.power_plant':          'energy.generation.power_plant',
    'energy.generation.solar_farm':           'energy.generation.solar_farm',
    'energy.generation.wind_farm':            'energy.generation.wind_farm',
    'water.wastewater.plant':                 'water.wastewater.plant',
    'water.treatment.plant':              'water.water_works',   # legacy manifest tag
    'water.water_works':                  'water.water_works',
    'water.storage_tank':                     'water.storage_tank',
    'transport.airport':                      'transport.airport',
    'transport.train_station':                'transport.train_station',
    'transport.port_terminal':                'transport.port_terminal',
    'telecom.data_center':                    'telecom.data_center',
}

SECTOR_TO_CLASS_IDX = {
    'energy':    [0, 1, 2, 3, 4, 5],
    'water':     [6, 7, 8],
    'transport': [9, 10, 11],
    'telecom':   [12],
}

# OlmoEarth band remap. our .npy storage:
#   [B04, B03, B02, B08, B8A, B11, B12, VV, VH]   indices 0..8
# we select the same 6 bands as Prithvi (order [B02, B03, B04, B8A, B11, B12])
# so the on-disk layout and percentile-normalize path are shared. the OlmoEarth
# backbone then expands to the 12-band S2 order and zero-pads unavailable bands.
OLMOEARTH_BAND_INDICES = [2, 1, 0, 4, 5, 6]

PERC_LO, PERC_HI = 2.0, 98.0
IMAGE_SIZE = 224
LP_EPOCHS  = 25
LP_BATCH   = 16
LP_LR      = 1e-3
WEIGHT_CAP = 10.0
SEEDS      = [314, 271, 161]


# ---- aggregate config -------------------------------------------------------
# full-protocol seeds. aggregate JSON write is gated on
# set(SEEDS) == set(FULL_PROTOCOL_SEEDS) so partial reruns don't overwrite
# a 3-seed aggregate.
FULL_PROTOCOL_SEEDS = [314, 271, 161]

def set_seed(seed):
    import random, numpy as np
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print(f'Output dir:           {OUTPUT_DIR}')
print(f'Split artifact:       {SPLIT_ARTIFACT_PATH}')
print(f'OlmoEarth band indices: {OLMOEARTH_BAND_INDICES}  (B02, B03, B04, B8A, B11, B12)')
print(f'Image size:           {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'Training seeds:       {SEEDS}')
print(f'Linear probe:         {LP_EPOCHS} epochs, batch {LP_BATCH}, lr {LP_LR}')


In [ ]:
import zipfile, shutil, time, re
from pathlib import Path

drive_path = Path(DATASETS_DRIVE)
MULTISECTOR_RE = re.compile(r'^dataset_([a-z-]+)_(energy|water|transport|telecom)_v1_1k$')


def discover_multisector_sources():
    sources, seen = [], set()
    for entry in sorted(drive_path.iterdir()):
        stem = entry.stem if entry.suffix == '.zip' else entry.name
        m = MULTISECTOR_RE.match(stem)
        if not m: continue
        region, sector = m.group(1), m.group(2)
        if region not in REGIONS: continue
        key = (region, sector)
        if key in seen: continue
        seen.add(key)
        sources.append((region, sector, entry, entry.suffix == '.zip'))
    return sources


def materialize_source(region, sector, src_path, is_zip, force=False):
    folder_name = f'dataset_{region}_{sector}_v1_1k'
    target_dir  = Path(DATASETS_LOCAL) / folder_name
    if not force and (target_dir / 'manifest.json').exists():
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [DONE]   {region:<22s} {sector:<10s} already present ({n} tiles)')
        return target_dir
    if not is_zip:
        if not (src_path / 'manifest.json').exists():
            return None
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.copytree(src_path, target_dir)
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [COPY]   {region:<22s} {sector:<10s} ({n} tiles)')
        return target_dir
    t0 = time.time()
    with zipfile.ZipFile(src_path) as zf:
        names = zf.namelist()
        wrapped_prefix = f'{folder_name}/'
        is_wrapped = any(n.startswith(wrapped_prefix) for n in names)
        if is_wrapped:
            zf.extractall(DATASETS_LOCAL)
        else:
            target_dir.mkdir(parents=True, exist_ok=True)
            zf.extractall(target_dir)
    n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
    print(f'  [EXTRACT]{region:<22s} {sector:<10s} {time.time()-t0:.0f}s ({n} tiles)')
    return target_dir


discovered = discover_multisector_sources()
ready = []
for region, sector, src_path, is_zip in discovered:
    local = materialize_source(region, sector, src_path, is_zip)
    if local and (local / 'manifest.json').exists() and any((local / 'images').glob('*.npy')):
        ready.append((region, sector, local))
print(f'\nReady: {len(ready)} (region, sector) pairs')


In [ ]:
import json
import numpy as np
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from collections import Counter
import random


@dataclass
class _Record:
    path: 'Path'
    asset_id: str
    asset_type: str


def percentile_normalize(arr, lo=PERC_LO, hi=PERC_HI):
    img = arr.astype(np.float32)
    low = np.percentile(img, lo); high = np.percentile(img, hi)
    if high <= low:
        return np.clip(img / 255.0, 0.0, 1.0)
    return np.clip((img - low) / (high - low), 0.0, 1.0)


class OlmoEarthDataset(Dataset):
    """Self-contained loader for a single `dataset_<region>_<sector>_v1_1k/`
    folder. Selects the 6 shared S2 bands per `OLMOEARTH_BAND_INDICES`, applies
    percentile_normalize, and unsqueezes a T=1 temporal dim so the default
    collate stacks to `(B, 6, 1, H, W)` (the FM backbone consumes
    the 5D input directly)."""
    def __init__(self, dataset_root,
                 band_indices=OLMOEARTH_BAND_INDICES,
                 allowed_asset_types=tuple(ASSET_TYPE_MAP.keys())):
        self.dataset_root = Path(dataset_root)
        self.band_indices = list(band_indices)
        self.allowed = set(allowed_asset_types)
        self.max_required_band = max(self.band_indices)

        manifest_path = self.dataset_root / 'manifest.json'
        if not manifest_path.exists():
            raise FileNotFoundError(f'missing manifest.json: {manifest_path}')
        with manifest_path.open() as f:
            manifest = json.load(f)
        records_in = manifest.get('records', [])
        images_dir = self.dataset_root / 'images'

        records, dropped = [], Counter()
        for r in records_in:
            at = r.get('asset_type')
            if not at or at not in self.allowed:
                dropped['filtered_type' if at else 'no_label'] += 1; continue
            img_file = r.get('image_file')
            if not img_file:
                dropped['no_image_file'] += 1; continue
            p = images_dir / img_file
            if not p.exists():
                dropped['missing_npy'] += 1; continue
            try:
                arr = np.load(p, mmap_mode='r')
                if arr.shape[0] < self.max_required_band + 1:
                    dropped['too_few_bands'] += 1; continue
            except Exception:
                dropped['load_error'] += 1; continue
            records.append(_Record(path=p, asset_id=str(r.get('asset_id', p.stem)),
                                   asset_type=at))
        if dropped:
            print(f'  OlmoEarthDataset({self.dataset_root.name}): dropped '
                  f'{sum(dropped.values())} records ({dict(dropped)})')
        if not records:
            raise RuntimeError(f'no usable records in {self.dataset_root}')
        self.records = records

    def __len__(self): return len(self.records)

    def _load_image(self, path):
        arr = np.load(path)
        arr = arr[self.band_indices, :, :]              # (6, H, W) in the shared S2 band order
        arr = percentile_normalize(arr)                  # -> [0, 1]
        return arr.astype(np.float32)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = self._load_image(r.path)                   # (6, H, W)
        t = torch.from_numpy(img).unsqueeze(1)           # (6, 1, H, W)  — T=1 dim
        return {'image': t, 'asset_id': r.asset_id, 'asset_type': r.asset_type}


class MultiSectorLabelWrapper(Dataset):
    def __init__(self, base, region, sector, input_size=IMAGE_SIZE):
        self.base = base; self.region = region; self.sector = sector
        self.input_size = input_size
        self.valid_indices, self.labels, self.asset_ids = [], [], []
        for i, r in enumerate(base.records):
            mapped = ASSET_TYPE_MAP.get(r.asset_type)
            if mapped is None: continue
            self.valid_indices.append(i)
            self.labels.append(CLASS_TO_IDX[mapped])
            self.asset_ids.append(r.asset_id)

    def __len__(self): return len(self.valid_indices)

    def __getitem__(self, idx):
        sample = self.base[self.valid_indices[idx]]
        img = sample['image']                            # (6, 1, H, W)
        C, T, H, W = img.shape
        img = img.reshape(C * T, 1, H, W)
        img = F.interpolate(img, size=(self.input_size, self.input_size),
                            mode='bilinear', align_corners=False)
        img = img.reshape(C, T, self.input_size, self.input_size)
        return {'image': img, 'label': self.labels[idx],
                'asset_id': sample['asset_id'],
                'region': self.region, 'sector': self.sector}


class SubsetView(Dataset):
    def __init__(self, base, indices):
        self.base = base; self.indices = indices
        self.region = base.region; self.sector = base.sector
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]


source_datasets = {}
for region, sector, local in ready:
    base = OlmoEarthDataset(local)
    ds = MultiSectorLabelWrapper(base, region=region, sector=sector)
    if len(ds): source_datasets[(region, sector)] = ds
print(f'Built {len(source_datasets)} cell datasets')


In [ ]:
# load the spatial split artifact and slice each cell into train/val/test.
asset_to_split = load_split_artifact(SPLIT_ARTIFACT_PATH)
print(f'Loaded split artifact: {len(asset_to_split):,} asset_id -> split entries')
print(f'  splits distribution: {Counter(asset_to_split.values())}')

splits = {}
n_unmapped = 0
for key, ds in source_datasets.items():
    region, sector = key
    tr_idx, va_idx, te_idx = [], [], []
    for i in range(len(ds)):
        asset_id = ds.asset_ids[i]
        sp = asset_to_split.get(asset_id)
        if sp == 'train':   tr_idx.append(i)
        elif sp == 'val':   va_idx.append(i)
        elif sp == 'test':  te_idx.append(i)
        else:                n_unmapped += 1
    splits[key] = {
        'train': SubsetView(ds, tr_idx),
        'val':   SubsetView(ds, va_idx),
        'test':  SubsetView(ds, te_idx),
    }

if n_unmapped > 0:
    print(f'NOTE: {n_unmapped} tiles have no split assignment (excluded from train/val/test).')

train_global = ConcatDataset([s['train'] for s in splits.values()])
val_global   = ConcatDataset([s['val']   for s in splits.values()])
test_global  = ConcatDataset([s['test']  for s in splits.values()])

print(f'\nGlobal: train={len(train_global)}  val={len(val_global)}  test={len(test_global)}')


In [ ]:
# diagnostic: regenerate the v1 random stratified split inside the
# notebook for an exact-match comparison vs the new spatial split.
print('=' * 76)
print('Diagnostic: train/val/test transition table (old random -> new spatial)')
print('=' * 76)


def old_stratified_split(dataset_labels, train_frac=0.7, val_frac=0.15, seed=42):
    by_class = {}
    for i, label in enumerate(dataset_labels):
        by_class.setdefault(label, []).append(i)
    rng = random.Random(seed)
    tr, va, te = [], [], []
    for cls in sorted(by_class):
        idxs = by_class[cls].copy()
        rng.shuffle(idxs)
        n = len(idxs); n_tr = int(n * train_frac); n_va = int(n * val_frac)
        tr.extend(idxs[:n_tr])
        va.extend(idxs[n_tr:n_tr + n_va])
        te.extend(idxs[n_tr + n_va:])
    return tr, va, te


old_split = {}
for key in sorted(source_datasets):
    ds = source_datasets[key]
    tr, va, te = old_stratified_split(ds.labels)
    for i in tr: old_split[ds.asset_ids[i]] = 'train'
    for i in va: old_split[ds.asset_ids[i]] = 'val'
    for i in te: old_split[ds.asset_ids[i]] = 'test'

common_ids = set(old_split) & set(asset_to_split)
print(f'Comparing on {len(common_ids):,} tiles in both old and new splits')
counts = Counter()
for aid in common_ids:
    counts[(old_split[aid], asset_to_split[aid])] += 1

print(f'\nold \\ new      train       val       test')
print('-' * 50)
for old_sp in ('train', 'val', 'test'):
    row = [counts.get((old_sp, new_sp), 0) for new_sp in ('train', 'val', 'test')]
    print(f'{old_sp:<8s}  {row[0]:>10d} {row[1]:>9d} {row[2]:>10d}')

same    = sum(counts.get((sp, sp), 0) for sp in ('train', 'val', 'test'))
changed = len(common_ids) - same
print(f'\nUnchanged: {same:,} ({100.0*same/len(common_ids):.1f}%)')
print(f'Changed:   {changed:,} ({100.0*changed/len(common_ids):.1f}%)')
print('\n(Expected ~47% changed — same spatial split as CROMA v2 / AE v2 / SatlasS2 v2 / SatlasS1 v2.)')


In [ ]:
import torch.nn as nn
from olmoearth_pretrain_minimal import load_model_from_id, ModelID, Normalizer
from olmoearth_pretrain_minimal.olmoearth_pretrain_v1.utils.constants import Modality
from olmoearth_pretrain_minimal.olmoearth_pretrain_v1.utils.datatypes import MaskedOlmoEarthSample


class OlmoEarthBackbone(nn.Module):
    """OlmoEarth v1.1-Base frozen backbone, mean-pooled patch features."""
    NAME = 'olmoearth_v1_1_base'
    HF_REPO = 'allenai/OlmoEarth-v1-Base'
    EXPECTED_FEATURE_DIM = 768  # ViT-Base

    def __init__(self, freeze=True):
        super().__init__()
        self.normalizer = Normalizer(std_multiplier=2.0)
        try:
            self.backbone = load_model_from_id(ModelID.OLMOEARTH_V1_1_BASE, load_weights=True)
            self._version = 'v1.1-Base'
        except AttributeError:
            self.backbone = load_model_from_id(ModelID.OLMOEARTH_V1_BASE, load_weights=True)
            self._version = 'v1-Base (fallback)'
            print('  WARNING: v1.1-Base not exposed by olmoearth_pretrain_minimal; using v1-Base')
        self.backbone.eval()
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False
        self.feature_dim = self._infer_feature_dim()

    def _prepare_input(self, x):
        B, C, T, H, W = x.shape
        assert C == 6 and T == 1, f'expected (B, 6, 1, H, W), got {x.shape}'
        twelve_band = torch.zeros(B, 12, T, H, W, device=x.device, dtype=x.dtype)
        twelve_band[:, 0] = x[:, 0]
        twelve_band[:, 1] = x[:, 1]
        twelve_band[:, 2] = x[:, 2]
        twelve_band[:, 7] = x[:, 3]
        twelve_band[:, 8] = x[:, 4]
        twelve_band[:, 9] = x[:, 5]
        
        # permute FIRST — moves bands from dim 1 to last dim: (B, H, W, T, 12)
        twelve_band = twelve_band.permute(0, 3, 4, 2, 1).contiguous()
        
        # then scale (after permute — order doesn't affect scaling, but keep it visually near normalize)
        twelve_band = twelve_band * 10000.0
        
        # then normalize
        arr = twelve_band.detach().cpu().numpy()
        arr = self.normalizer.normalize(Modality.SENTINEL2_L2A, arr)
        
        return torch.from_numpy(arr).to(device=x.device, dtype=torch.float32)

    def _build_sample(self, s2_tensor):
        """Wrap normalized S2 tensor in MaskedOlmoEarthSample with required timestamps + masks."""
        B, H, W, T, D = s2_tensor.shape   # (B, H, W, T, 12)

        # timestamps: (B, T, 3) with [year_offset, month_0_to_11, day_offset]
        timestamps = torch.zeros(B, T, 3, dtype=torch.long, device=s2_tensor.device)
        timestamps[:, :, 1] = 6  # month = June

        # modality mask: pixel-resolution bool tensor. False = visible token.
        # for feature extraction (LP, no MIM pretraining), we want no masking:
        # all tokens visible → all-False mask.
        s2_mask = torch.zeros(B, H, W, T, dtype=torch.bool, device=s2_tensor.device)

        sample = MaskedOlmoEarthSample(
            timestamps=timestamps,
            sentinel2_l2a=s2_tensor,
            sentinel2_l2a_mask=s2_mask,
        )
        return sample

    def _infer_feature_dim(self):
        device = next(self.backbone.parameters()).device
        dummy = torch.zeros(1, 6, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)
        with torch.no_grad():
            feat = self.forward(dummy)
        d = feat.shape[-1]
        print(f'  feature_dim = {d}')
        return d

    def forward(self, x):
        prepared = self._prepare_input(x)
        sample = self._build_sample(prepared)
        out = self.backbone.encoder(sample, patch_size=8)
        return self._extract_features(out)

    @staticmethod
    def _extract_features(out):
        """Extract a pooled (B, D) feature vector from OlmoEarth's encoder output dict.
        
        OlmoEarth v1.1-Base encoder returns:
        - 'project_aggregated': (B, 768) — the pooled global feature. THIS is what we want.
        - 'tokens_and_masks': TokensAndMasks object holding per-token features + masks
            (useful for decoder pretraining; not needed for LP).
        """
        if isinstance(out, dict) and 'project_aggregated' in out:
            return out['project_aggregated']
        # fallbacks in case the library API shifts in future versions
        if isinstance(out, dict):
            for key in ('latent_projected_and_pooled', 'pooled', 'cls', 'global_feat'):
                if key in out and isinstance(out[key], torch.Tensor) and out[key].dim() == 2:
                    return out[key]
            raise RuntimeError(f'Cannot extract feature from encoder dict: keys={list(out.keys())}')
        if hasattr(out, 'shape'):
            if out.dim() == 2:
                return out
            if out.dim() == 3:
                return out.mean(dim=1)
            if out.dim() == 4:
                return out.mean(dim=[1, 2])
        raise RuntimeError(f'Unexpected encoder output: {type(out).__name__}')


class InfraBenchClassifier(nn.Module):
    def __init__(self, backbone, num_classes, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(nn.Dropout(dropout),
                                  nn.Linear(backbone.feature_dim, num_classes))
    def forward(self, x):
        return self.head(self.backbone(x))


def BACKBONE_FACTORY(freeze=True):
    return OlmoEarthBackbone(freeze=freeze)


print('OlmoEarthBackbone + InfraBenchClassifier defined.')


In [ ]:
set_seed(SEEDS[0])
sb = OlmoEarthBackbone(freeze=True)

device = next(sb.parameters()).device
samples = [val_global[i] for i in range(4)]
x_batch = torch.stack([s['image'] for s in samples]).to(device)

with torch.no_grad():
    final_feat = sb(x_batch)

print(f"Final feature variance across samples: {final_feat.var(dim=0).mean().item():.6f}")
print(f"Norms per sample: {final_feat.norm(dim=-1).tolist()}")

In [ ]:
from torch.optim import AdamW
from collections import defaultdict
from sklearn.metrics import f1_score, confusion_matrix
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


def _labels_from(dataset):
    if isinstance(dataset, ConcatDataset):
        for d in dataset.datasets:
            yield from _labels_from(d)
    elif isinstance(dataset, SubsetView):
        for i in dataset.indices:
            yield dataset.base.labels[i]
    else:
        yield from dataset.labels


def compute_class_weights(train_set, max_weight=WEIGHT_CAP):
    counts = Counter(_labels_from(train_set))
    total = sum(counts.values())
    weights = torch.zeros(len(CLASS_NAMES))
    for c in range(len(CLASS_NAMES)):
        if counts.get(c, 0) > 0:
            w = total / (len(CLASS_NAMES) * counts[c])
            weights[c] = min(w, max_weight)
    return weights


def collate(batch):
    return {
        'image':  torch.stack([b['image'] for b in batch]),   # (B, 6, 1, H, W)
        'label':  torch.tensor([b['label'] for b in batch], dtype=torch.long),
        'region': [b['region'] for b in batch],
        'sector': [b['sector'] for b in batch],
    }


def _per_sector_v2(y_true, y_pred, per_class_f1, cm):
    """v2 per-sector — matches CROMA v2 schema exactly.
    Mean of per-class F1s for the classes in that sector, computed on the
    FULL test set."""
    cm = np.array(cm)
    out = {}
    for sector, class_idxs in SECTOR_TO_CLASS_IDX.items():
        sector_f1s = [float(per_class_f1[i]) for i in class_idxs]
        macro_f1 = float(np.mean(sector_f1s)) if sector_f1s else 0.0
        n_sector = int(sum(cm[i, :].sum() for i in class_idxs))
        correct  = int(sum(cm[i, i]      for i in class_idxs))
        acc = correct / n_sector if n_sector > 0 else float('nan')
        out[sector] = {
            'n': n_sector,
            'macro_f1': macro_f1,
            'acc': acc,
            'per_class_f1_in_sector': {
                CLASS_NAMES[i]: sector_f1s[idx]
                for idx, i in enumerate(class_idxs)
            },
        }
    return out


@torch.no_grad()
def evaluate(model, loader, return_breakdowns=False):
    model.eval()
    all_preds, all_labels, all_regions, all_sectors = [], [], [], []
    for batch in loader:
        logits = model(batch['image'].to(DEVICE, non_blocking=True))
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy().tolist())
        all_regions.extend(batch['region'])
        all_sectors.extend(batch['sector'])

    per_class_f1 = f1_score(all_labels, all_preds, average=None,
                            labels=list(range(len(CLASS_NAMES))),
                            zero_division=0.0).tolist()
    cm = confusion_matrix(all_labels, all_preds,
                          labels=list(range(len(CLASS_NAMES)))).tolist()
    result = {
        'acc': float(np.mean(np.array(all_preds) == np.array(all_labels))),
        'macro_f1': float(f1_score(all_labels, all_preds, average='macro', zero_division=0.0)),
        'per_class_f1': per_class_f1,
        'confusion': cm,
    }
    if return_breakdowns:
        def grouped_region_f1():
            groups = defaultdict(lambda: {'preds': [], 'labels': []})
            for p, l, g in zip(all_preds, all_labels, all_regions):
                groups[g]['preds'].append(p)
                groups[g]['labels'].append(l)
            return {
                g: {
                    'n': len(d['labels']),
                    'macro_f1': float(f1_score(d['labels'], d['preds'], average='macro',
                                               zero_division=0.0)),
                    'acc': float(np.mean(np.array(d['preds']) == np.array(d['labels']))),
                }
                for g, d in groups.items()
            }
        result['per_region']                 = grouped_region_f1()
        result['per_sector']                 = _per_sector_v2(all_labels, all_preds, per_class_f1, cm)
        result['per_sector_corrected']       = result['per_sector']
        result['per_sector_correction_note'] = (
            'per_sector = macro-average of per-class F1s for classes in that '
            'sector, computed on the FULL test set (v2 definition).'
        )
    return result


def train_one_seed(seed, *, train_set, val_set, test_set):
    set_seed(seed)
    print(f'\n--- seed {seed} ---')
    backbone = BACKBONE_FACTORY(freeze=True)
    model = InfraBenchClassifier(backbone, num_classes=len(CLASS_NAMES)).to(DEVICE)

    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_set, batch_size=LP_BATCH, shuffle=True,
                              num_workers=2, collate_fn=collate, pin_memory=True,
                              generator=g)
    val_loader   = DataLoader(val_set,   batch_size=LP_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=LP_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)

    weights = compute_class_weights(train_set).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad],
                      lr=LP_LR, weight_decay=1e-4)

    run_name = f'olmoearth_lp_v1_seed{seed}_linear_probe'
    # per-epoch resume checkpoint + metrics stream (flat files at OUTPUT_DIR).
    resume_ckpt_path = Path(OUTPUT_DIR) / f'{run_name}_checkpoint.pt'
    metrics_jsonl    = Path(OUTPUT_DIR) / f'{run_name}_metrics.jsonl'

    # ---- resume-from-checkpoint (mirrors the CROMA pattern) -----------------
    history        = []
    best_val_f1    = -1.0
    best_epoch     = -1
    best_val_state = None   # best-so-far model state (in-memory, for BEST_CKPT_BEFORE_TEST)
    start_epoch    = 0

    if resume_ckpt_path.exists():
        ckpt = torch.load(resume_ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        history        = ckpt.get('history', [])
        best_val_f1    = float(ckpt.get('best_val_f1', -1.0))
        best_epoch     = int(ckpt.get('best_epoch', -1))
        best_val_state = ckpt.get('best_val_state_dict', None)
        start_epoch    = int(ckpt.get('epoch', 0))

        # rewrite metrics.jsonl from checkpoint history (checkpoint is
        # authoritative; eliminates the metrics-vs-checkpoint race).
        with metrics_jsonl.open('w', encoding='utf-8') as _mf:
            for _entry in history:
                _mf.write(json.dumps(_entry) + '\n')
            _mf.flush()
            os.fsync(_mf.fileno())

        print(f'  found checkpoint: {resume_ckpt_path.name} (epoch {start_epoch} completed)')
        print(f'  best_val_f1 so far: {best_val_f1:.4f} at epoch {best_epoch}')
        print(f'  history length: {len(history)}')
        print(f'  rewrote metrics.jsonl from checkpoint history')
        print(f'  resuming from epoch {start_epoch + 1}')
    else:
        # fresh start: create empty metrics.jsonl.
        metrics_jsonl.write_text('', encoding='utf-8')
        print(f'  no checkpoint found, starting fresh at epoch 1')

    for epoch in range(start_epoch, LP_EPOCHS):
        model.train()
        t0 = time.time()
        loss_sum, n = 0.0, 0
        for batch in train_loader:
            images = batch['image'].to(DEVICE, non_blocking=True)
            labels = batch['label'].to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * images.size(0)
            n += images.size(0)
        train_loss = loss_sum / max(n, 1)
        val = evaluate(model, val_loader)
        entry = {
            'epoch': epoch + 1, 'train_loss': train_loss,
            'val_acc': val['acc'], 'val_macro_f1': val['macro_f1'],
            'time_s': time.time() - t0,
        }
        history.append(entry)

        # append entry to metrics.jsonl + fsync (durable line-per-epoch stream).
        with metrics_jsonl.open('a', encoding='utf-8') as _mf:
            _mf.write(json.dumps(entry) + '\n')
            _mf.flush()
            os.fsync(_mf.fileno())

        marker = ''
        if val['macro_f1'] > best_val_f1:
            best_val_f1 = val['macro_f1']
            best_epoch = epoch + 1
            marker = ' *'
            # keep best state in memory (moved to CPU to keep GPU free)
            best_val_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}

        # per-epoch resume checkpoint. written AFTER metrics append so the
        # checkpoint's `history` field is a strict superset-or-equal of what's
        # in metrics.jsonl; on resume we rewrite metrics.jsonl from history.
        torch.save({
            'epoch':               epoch + 1,   # last completed epoch (1-indexed)
            'model_state_dict':    model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history':             history,
            'best_val_f1':         best_val_f1,
            'best_epoch':          best_epoch,
            'best_val_state_dict': best_val_state,
        }, resume_ckpt_path)

        print(f'  ep {epoch+1:>3d}  loss={train_loss:.4f}  '
              f'val_acc={val["acc"]:.4f}  val_f1={val["macro_f1"]:.4f}{marker}')

    # ============== BEST_CKPT_BEFORE_TEST =================================
    if best_val_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_val_state.items()})
        print(f'  [BEST_CKPT_BEFORE_TEST] restored best-val state from epoch {best_epoch} '
              f'(val_f1={best_val_f1:.4f}) before test')
        tested_with = f'best-val checkpoint from epoch {best_epoch}'
    else:
        tested_with = 'final-epoch state'

    tail = [h['val_macro_f1'] for h in history[-min(5, len(history)):]]
    test = evaluate(model, test_loader, return_breakdowns=True)

    # cleanup: delete resume checkpoint (final JSON is authoritative).
    # metrics.jsonl is left in place for post-hoc training-curve analysis.
    if resume_ckpt_path.exists():
        try:
            resume_ckpt_path.unlink()
            print(f'  cleaned up resume checkpoint: {resume_ckpt_path.name}')
        except OSError as e:
            print(f'  WARNING: failed to delete resume checkpoint: {e}')

    return {
        'run_name':     run_name,
        'backbone':     backbone.NAME,
        'condition':    'linear_probe',
        'num_epochs':   LP_EPOCHS,
        'seed':         seed,
        'best_val_f1':  best_val_f1,
        'best_epoch':   best_epoch,
        'tail_mean_f1': float(np.mean(tail)),
        'tail_std_f1':  float(np.std(tail)),
        'history':      history,
        'test':         test,
        'tested_with':  tested_with,
    }


print('Training infrastructure ready (v2 per-sector F1, BEST_CKPT_BEFORE_TEST, multi-seed).')


In [ ]:
!pip install -q hf_transfer
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# then retry the safetensors download
from huggingface_hub import hf_hub_download
import time

print("Downloading with hf_transfer enabled...")
start = time.time()
try:
    path = hf_hub_download(
        repo_id='facebook/dinov3-vitl16-pretrain-lvd1689m',
        filename='model.safetensors',
    )
    elapsed = time.time() - start
    print(f"SUCCESS in {elapsed:.1f}s: {path}")
except Exception as e:
    elapsed = time.time() - start
    print(f"FAILED after {elapsed:.1f}s: {type(e).__name__}: {e}")

In [ ]:
# default SMOKE_ONLY=True. if smoke fails at any specific point, that
# failure IS the diagnostic. flip to False once smoke is happy.
SMOKE_ONLY = False

print('Loading OlmoEarth backbone...')
set_seed(SEEDS[0])
try:
    sb = OlmoEarthBackbone(freeze=True)
except Exception as e:
    print()
    print('=' * 76)
    print('SMOKE CHECK FAILED at backbone load.')
    print('=' * 76)
    print('See the traceback above. Per the v2 plan, this failure point is')
    print("the signal we needed -- it tells us whether OlmoEarth inclusion is")
    print('feasible for v1 paper scope or genuinely needs to be deferred.')
    raise

sm = InfraBenchClassifier(sb, num_classes=len(CLASS_NAMES)).to(DEVICE)
print(f'  loader_used = {getattr(sb, "_loader_used", getattr(sb, "_version", "n/a"))}')
print(f'  feature_dim = {sb.feature_dim}')
print(f'  trainable params = {sum(p.numel() for p in sm.parameters() if p.requires_grad):,}')

smoke_loader = DataLoader(train_global, batch_size=4, shuffle=False,
                          num_workers=0, collate_fn=collate)
batch = next(iter(smoke_loader))
img = batch['image']
print(f'\n  Batch shape: {tuple(img.shape)}  (expect [4, 6, 1, {IMAGE_SIZE}, {IMAGE_SIZE}])')
print(f'  Batch range: [{img.min().item():.4f}, {img.max().item():.4f}]  (expect ~[0, 1])')

print('\nForward pass...')
sm.eval()
with torch.no_grad():
    feats = sb(img.to(DEVICE))
    logits = sm(img.to(DEVICE))
print(f'  Backbone features: {tuple(feats.shape)}  (expect [4, {sb.feature_dim}])')
print(f'  Classifier logits: {tuple(logits.shape)}  (expect [4, {len(CLASS_NAMES)}])')
print(f'  Logits finite: {torch.isfinite(logits).all().item()}')
assert torch.isfinite(logits).all().item(), 'NaN/Inf in logits — aborting smoke'

print('\nOne training step...')
sm.train()
weights = compute_class_weights(train_global).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = AdamW([p for p in sm.parameters() if p.requires_grad],
                  lr=LP_LR, weight_decay=1e-4)
optimizer.zero_grad()
loss = criterion(sm(img.to(DEVICE)), batch['label'].to(DEVICE))
loss.backward()
head_grads = [p.grad for p in sm.head.parameters() if p.grad is not None]
assert head_grads and any(g.abs().sum().item() > 0 for g in head_grads), \
    'head received zero gradients'
optimizer.step()
print(f'  Train step loss: {loss.item():.4f}  (finite={torch.isfinite(loss).item()})')

print(f'\nSmoke check PASSED. SMOKE_ONLY = {SMOKE_ONLY} — multi-seed cell below will '
      f'{"NOT train" if SMOKE_ONLY else "run all 3 seeds"}.')


In [ ]:
import json as _json
import numpy as np

if SMOKE_ONLY:
    print('SMOKE_ONLY=True — skipping all training. Set False and re-run.')
else:
    per_seed_results = {}
    for seed in SEEDS:
        # --- skip-if-JSON-exists guard ---
        # if this seed's per-seed JSON already exists on Drive, load it and skip
        # retraining. also cleans up any stale resume checkpoint for this seed.
        out_path = Path(OUTPUT_DIR) / f'olmoearth_lp_v1_seed{seed}_results.json'
        stale_ckpt = Path(OUTPUT_DIR) / f'olmoearth_lp_v1_seed{seed}_linear_probe_checkpoint.pt'
        if out_path.exists():
            if stale_ckpt.exists():
                try:
                    stale_ckpt.unlink()
                    print(f'  [CLEANUP] deleted stale checkpoint: {stale_ckpt.name}')
                except OSError as e:
                    print(f'  WARNING: failed to delete stale checkpoint: {e}')
            with out_path.open() as f:
                per_seed_results[seed] = _json.load(f)['linear_probe']
            print(f'  [SKIP] seed {seed}: existing per-seed JSON at '
                  f'{out_path.name} (delete to force rerun)')
            continue
        if stale_ckpt.exists():
            print(f'  [RESUME] seed {seed}: found existing checkpoint '
                  f'{stale_ckpt.name} -- train_one_seed will resume from it')
        result = train_one_seed(seed,
                                train_set=train_global,
                                val_set=val_global,
                                test_set=test_global)
        per_seed_results[seed] = result
        with open(out_path, 'w') as f:
            _json.dump({'linear_probe': result}, f, indent=2)
        print(f'\n  saved {out_path}')
    def _agg(values):
        arr = np.array(values, dtype=np.float64)
        return {'mean': float(arr.mean()), 'std': float(arr.std(ddof=0)),
                'per_seed': [float(v) for v in arr]}

    agg = {}
    agg['test_macro_f1'] = _agg([per_seed_results[s]['test']['macro_f1'] for s in SEEDS])
    agg['test_accuracy'] = _agg([per_seed_results[s]['test']['acc'] for s in SEEDS])

    per_class_arr = np.array([per_seed_results[s]['test']['per_class_f1'] for s in SEEDS])
    agg['per_class_f1'] = [
        {'class': CLASS_NAMES[i], 'idx': i,
         'mean_f1': float(per_class_arr[:, i].mean()),
         'std_f1':  float(per_class_arr[:, i].std(ddof=0)),
         'per_seed': [float(v) for v in per_class_arr[:, i]]}
        for i in range(len(CLASS_NAMES))
    ]

    agg['per_sector_f1'] = {}
    for sector in SECTOR_TO_CLASS_IDX:
        f1s = [per_seed_results[s]['test']['per_sector'][sector]['macro_f1'] for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_sector'][sector]['n']
        agg['per_sector_f1'][sector] = {
            'n': n_seed,
            'mean_macro_f1': float(np.mean(f1s)),
            'std_macro_f1':  float(np.std(f1s, ddof=0)),
            'per_seed': [float(v) for v in f1s],
        }

    region_keys = sorted({r for s in SEEDS for r in per_seed_results[s]['test']['per_region']})
    agg['per_region_f1'] = {}
    for region in region_keys:
        f1s = [per_seed_results[s]['test']['per_region'].get(region, {}).get('macro_f1', float('nan'))
               for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_region'].get(region, {}).get('n', 0)
        agg['per_region_f1'][region] = {
            'n': n_seed,
            'mean_macro_f1': float(np.nanmean(f1s)),
            'std_macro_f1':  float(np.nanstd(f1s, ddof=0)),
            'per_seed': [float(v) for v in f1s],
        }

    agg['seeds']                       = SEEDS
    agg['split_artifact']              = SPLIT_ARTIFACT_PATH
    agg['per_sector_correction_note']  = per_seed_results[SEEDS[0]]['test']['per_sector_correction_note']

    _is_full_run = set(SEEDS) == set(FULL_PROTOCOL_SEEDS)
    if _is_full_run:
        agg_path = Path(OUTPUT_DIR) / 'olmoearth_lp_v1_aggregate.json'
        with open(agg_path, 'w') as f:
            _json.dump(agg, f, indent=2)
        print(f'\nAggregate saved: {agg_path}')
    else:
        print(f'\nNOTE: SEEDS = {SEEDS}, not the full protocol '
              f'{FULL_PROTOCOL_SEEDS}. Skipping aggregate JSON write '
              f'to preserve any existing 3-seed aggregate.')

    # confusion matrix PNG
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    cm_sum = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=np.int64)
    for s in SEEDS:
        cm_sum += np.array(per_seed_results[s]['test']['confusion'], dtype=np.int64)
    cm_norm = cm_sum.astype(np.float64)
    row_sums = cm_norm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm_norm, row_sums, out=np.zeros_like(cm_norm), where=row_sums > 0)

    short_names = [n.split('.', 1)[1] if '.' in n else n for n in CLASS_NAMES]
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm_norm, cmap='Oranges', vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(short_names, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    mf1 = agg['test_macro_f1']
    ax.set_title(f'OlmoEarth v1.1-Base LP -- aggregate confusion (summed over '
                 f'{len(SEEDS)} seeds, row-normalized)\n'
                 f'macro F1 = {mf1["mean"]:.3f} +/- {mf1["std"]:.3f}, seeds = {SEEDS}')
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            v = cm_norm[i, j]
            if v > 0.01:
                color = 'white' if v > 0.5 else 'black'
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', color=color, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    if _is_full_run:
        cm_path = Path(OUTPUT_DIR) / 'confusion_matrix_olmoearth_lp_v1_aggregate.png'
        plt.savefig(cm_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f'Confusion matrix saved: {cm_path}')

    print('\n' + '=' * 76)
    print(f'OlmoEarth v1.1-Base LP -- aggregate ({len(SEEDS)} seeds)')
    print('=' * 76)
    print(f'Test macro F1: {agg["test_macro_f1"]["mean"]:.4f} +/- '
          f'{agg["test_macro_f1"]["std"]:.4f}')
    print(f'Test accuracy: {agg["test_accuracy"]["mean"]:.4f} +/- '
          f'{agg["test_accuracy"]["std"]:.4f}')
    print('\nPer-class F1 (mean +/- std):')
    for entry in agg['per_class_f1']:
        print(f'  [{entry["idx"]:>2d}] {entry["class"]:<34s} '
              f'{entry["mean_f1"]:.4f} +/- {entry["std_f1"]:.4f}')
    print('\nPer-sector F1 (v2):')
    for sector, stats in agg['per_sector_f1'].items():
        print(f'  {sector:<10s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')
    print('\nPer-region F1:')
    for region, stats in agg['per_region_f1'].items():
        print(f'  {region:<22s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')
